# Imports & Setup

In [ ]:
%matplotlib inline

import logging
import os
from datetime import datetime

import pandas as pd
from tqdm import tqdm

from metrics import MetricsContainer
from loading import load_experiment

from analytics.training import plot_models_together

log = logging.getLogger(__name__)

# Experiments Loading

In [ ]:
log_root = "..\logs\ByModel"
output_dir = log_root
data = load_experiment(log_root, load="final")
log.info("Loaded data for %s model configurations.", len(data))

In [ ]:
models = {}

for key, seed_dict in data.items():
    normalized = key.replace("/", os.sep).replace("\\", os.sep)
    parts = [p for p in normalized.split(os.sep) if p]
    if not parts: continue
    family = parts[0].strip().lower()
    architecture = parts[1].strip().lower() if len(parts) > 1 else ""
    variant = os.sep.join(parts[2:]) if len(parts) > 2 else ""
    
    models.setdefault(family, {}).setdefault(architecture, {})[variant] = seed_dict

len(models.get("classical", {})), len(models.get("hybrid", {})), len(models.get("quantum", {}))

### Manual Selections

In [ ]:
classical_baselines = {
    "MLP": models['classical']['mlp']['hidden_sizes=[1024, 512, 64, 64]'],
    "ConvNet": models['classical']['convnet']['conv_channels=[32,64],fc_sizes=[128]'],
    "ResNet18": models['classical']['resnet18'][''],
    "MLP Mixer": models['classical']['mlpmixer']['']
}

hybrid_baselines = {
    "HPVQL": models['hybrid']['hpvql'][''],
    "HPVQM": models['hybrid']['hpvqm']['qblocks=5,qdepth=50'],
    "HQE": models['hybrid']['hqe'][''],
    "HQUANV": models['hybrid']['hquanv'][''],
    "HVQL": models['hybrid']['hvql']['ansatz=strong,qdepth=50'],
    "HVQM": models['hybrid']['hvqm']['ansatz=strong,qdepth=50']
}

# Training Figures

## Classical Figures

In [ ]:
# Comparison of classical baselines
plot_models_together(classical_baselines, fig_dir=f"{output_dir}/classical", show=True)

In [ ]:
# ConvNet Variants

convnet_models = {}
convnet_models.update(models['classical']['convnet'])
convnet_models.update({'lenet':models['classical']['lenet']['']})

plot_models_together(convnet_models, fig_dir=f"{output_dir}/classical/convnet", show=True)

In [ ]:
# Combine classical MLP and LeNet variants into one dict

plot_models_together(models['classical']['mlp'], fig_dir=f"{output_dir}/classical/mlp", show=True)

## Hybrid Figures

In [ ]:
# Comparison of hybrid baselines
plot_models_together(hybrid_baselines, fig_dir=f"{output_dir}/hybrid", show=True)

In [ ]:
hvql_no_basic = {k: v for k, v in models['hybrid']['hvql'].items() if "basic" not in k.lower()}

hvql_no_basic.keys()

In [ ]:
# The different quantum architectures with their variants

# Removing broken basic variants of HVQL for better visualization
hvql_no_basic = {k: v for k, v in models['hybrid']['hvql'].items() if "basic" not in k.lower()}

plot_models_together(models['hybrid']['hpvqm'], fig_dir=f"{output_dir}/hybrid/hpvqm", show=True)
plot_models_together(hvql_no_basic, fig_dir=f"{output_dir}/hybrid/hvql", show=True)
plot_models_together(models['hybrid']['hvqm'], fig_dir=f"{output_dir}/hybrid/hvqm", show=True)

## Quantum Models

In [ ]:
models['quantum']['vqm'].keys()

In [ ]:
# All versions
# vqm_filtered = {k: v for k, v in models['quantum']['vqm'].items() if k != "VQM50-Basic" and k != "VQM50-identity" and k != "VQM50-NoEntangling"}

plot_models_together(models['quantum']['vqm'], fig_dir=f"{output_dir}/quantum/vqm", show=True)

In [ ]:
# Depth variants for VQM
vqm_depth_variants = {
    "VQM50-Strong": models['quantum']['vqm']['VQM50-Strong'],
    "VQM100": models['quantum']['vqm']['VQM100'],
    "VQM200": models['quantum']['vqm']['VQM200'],
}

# TODO Need to update file name
plot_models_together(vqm_depth_variants, fig_dir=f"{output_dir}/quantum/vqm", show=True)

In [ ]:
# Ansatz variants for VQM
vqm_ansatz_variants = {
    "VQM50-Basic": models['quantum']['vqm']['VQM50-Basic'],
    "VQM50-Identity": models['quantum']['vqm']['VQM50-Identity'],
    "VQM50-NoEntangling": models['quantum']['vqm']['VQM50-NoEntangling'],
    "VQM50-Random": models['quantum']['vqm']['VQM50-Random'],
    "VQM50-Strong": models['quantum']['vqm']['VQM50-Strong'],
}

# TODO Need to update file name
plot_models_together(vqm_ansatz_variants, fig_dir=f"{output_dir}/quantum/vqm", show=True)